In [1]:
import pandas as pd
import numpy as np

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

In [3]:
df = pd.read_csv("../data/processed/paysim_features.csv")

df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,...,origin_balance_zero,destination_balance_zero,full_origin_balance_transfer,destination_previous_transactions,origin_previous_transactions,origin_balance_depleted,destination_balance_change,origin_balance_change,amount_to_destination_balance,zero_amount_transaction
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,...,0,1,0,0,0,0,0.0,9839.64,NaN,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,...,0,1,0,0,0,0,0.0,1864.28,NaN,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,...,0,1,1,0,0,1,0.0,181.00,NaN,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,...,0,0,1,0,0,1,-21182.0,181.00,0.008545,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,...,0,1,0,0,0,0,0.0,11668.14,NaN,0


In [4]:
df.shape

(6362620, 24)

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 24 columns):
 #   Column                             Dtype  
---  ------                             -----  
 0   step                               int64  
 1   type                               str    
 2   amount                             float64
 3   nameOrig                           str    
 4   oldbalanceOrg                      float64
 5   newbalanceOrig                     float64
 6   nameDest                           str    
 7   oldbalanceDest                     float64
 8   newbalanceDest                     float64
 9   isFraud                            int64  
 10  log_amount                         float64
 11  hour                               int64  
 12  day                                int64  
 13  origin_balance_utilization         float64
 14  origin_balance_zero                int64  
 15  destination_balance_zero           int64  
 16  full_origin_balance_transfer 

In [6]:
X = df.drop(columns=["isFraud"])
y = df["isFraud"]

In [7]:
print(y.value_counts())
print("\nFraud percentage:")
print((y.mean() * 100).round(6))

isFraud
0    6354407
1       8213
Name: count, dtype: int64

Fraud percentage:
0.129082


In [8]:
# Remove identifier columns
X = X.drop(columns=["nameOrig", "nameDest"])

print("Features remaining:", X.shape[1])
print("\nRemaining columns:")
print(X.columns.tolist())

Features remaining: 21

Remaining columns:
['step', 'type', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'log_amount', 'hour', 'day', 'origin_balance_utilization', 'origin_balance_zero', 'destination_balance_zero', 'full_origin_balance_transfer', 'destination_previous_transactions', 'origin_previous_transactions', 'origin_balance_depleted', 'destination_balance_change', 'origin_balance_change', 'amount_to_destination_balance', 'zero_amount_transaction']


In [9]:
## Trai test split with 70% train / 15% val / 15% test split.
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

In [10]:
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

In [11]:
print("\nFraud percentage:")
print(f"Train:      {y_train.mean() * 100:.4f}%")
print(f"Validation: {y_val.mean() * 100:.4f}%")
print(f"Test:       {y_test.mean() * 100:.4f}%")


Fraud percentage:
Train:      0.1291%
Validation: 0.1291%
Test:       0.1291%


In [12]:
## Standardization using OneHotEncoder, ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

In [13]:
categorical_features = ['type']

numerical_features = [feature for feature in X_train.columns if feature not in categorical_features]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "numerical",
            "passthrough",
            numerical_features
        )
    ]
)

print("Categorical features:", categorical_features)
print("Numerical features:", len(numerical_features))

Categorical features: ['type']
Numerical features: 20


In [14]:
# Transformation
X_train_processed = preprocessor.fit_transform(X_train)

X_val_processed = preprocessor.transform(X_val)

X_test_processed = preprocessor.transform(X_test)

In [15]:
print("Original features:", X_train.shape[1])
print("Processed features:", X_train_processed.shape[1])

print("\nTrain:", X_train_processed.shape)
print("Validation:", X_val_processed.shape)
print("Test:", X_test_processed.shape)

Original features: 21
Processed features: 25

Train: (4453834, 25)
Validation: (954393, 25)
Test: (954393, 25)


In [16]:
negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()

scale_pos_weight = negative_count / positive_count

print("Legitimate transactions:", negative_count)
print("Fraudulent transactions:", positive_count)
print("Scale Pos Weight:", scale_pos_weight)

Legitimate transactions: 4448085
Fraudulent transactions: 5749
Scale Pos Weight: 773.7145590537485


In [18]:
## Importing XGBClassifier
from xgboost import XGBClassifier


In [19]:
xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

In [20]:
xgb_model.fit(
    X_train_processed,
    y_train,
    eval_set=[(X_train_processed, y_train), (X_val_processed, y_val)],
    verbose=50
)

[0]	validation_0-aucpr:0.40337	validation_1-aucpr:0.39544
[50]	validation_0-aucpr:0.99970	validation_1-aucpr:0.99862
[100]	validation_0-aucpr:0.99999	validation_1-aucpr:0.99885
[150]	validation_0-aucpr:1.00000	validation_1-aucpr:0.99866
[200]	validation_0-aucpr:1.00000	validation_1-aucpr:0.99836
[250]	validation_0-aucpr:1.00000	validation_1-aucpr:0.99818
[299]	validation_0-aucpr:1.00000	validation_1-aucpr:0.99806


,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'aucpr'
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [21]:
# ============================================================
# 12. GENERATE VALIDATION FRAUD PROBABILITIES
# ============================================================

y_val_proba = xgb_model.predict_proba(X_val_processed)[:, 1]

print("Number of validation predictions:", len(y_val_proba))
print("Minimum probability:", y_val_proba.min())
print("Maximum probability:", y_val_proba.max())
print("Mean probability:", y_val_proba.mean())

Number of validation predictions: 954393
Minimum probability: 1.3092515e-10
Maximum probability: 1.0
Mean probability: 0.0012942341


In [ ]:
# ============================================================
# 13. BASELINE CLASSIFICATION — THRESHOLD = 0.50
# ============================================================

# Convert predicted probabilities into class predictions
# using the default threshold of 0.50

y_val_pred = (y_val_proba >= 0.50).astype(int)

print("Predicted class distribution:")
print(pd.Series(y_val_pred).value_counts())


Predicted class distribution:
0    953161
1      1232
Name: count, dtype: int64


In [23]:
# ============================================================
# 14. CLASSIFICATION REPORT — THRESHOLD = 0.50
# ============================================================

print(classification_report(
    y_val,
    y_val_pred,
    target_names=["Legitimate", "Fraud"],
    digits=4
))

              precision    recall  f1-score   support

  Legitimate     1.0000    1.0000    1.0000    953161
       Fraud     0.9968    0.9968    0.9968      1232

    accuracy                         1.0000    954393
   macro avg     0.9984    0.9984    0.9984    954393
weighted avg     1.0000    1.0000    1.0000    954393



In [25]:
cm = confusion_matrix(y_val, y_val_pred)

print("Confusion Matrix:", cm)

print("-"*50)

tn, fp, fn, tp = cm.ravel()

print(f"True Negatives  (TN): {tn:,}")
print(f"False Positives (FP): {fp:,}")
print(f"False Negatives (FN): {fn:,}")
print(f"True Positives  (TP): {tp:,}")

Confusion Matrix: [[953157      4]
 [     4   1228]]
--------------------------------------------------
True Negatives  (TN): 953,157
False Positives (FP): 4
False Negatives (FN): 4
True Positives  (TP): 1,228


In [26]:
# ============================================================
# 17. ROC-AUC AND PR-AUC
# ============================================================

roc_auc = roc_auc_score(y_val, y_val_proba)

pr_auc = average_precision_score(y_val, y_val_proba)

print(f"ROC-AUC: {roc_auc:.6f}")
print(f"PR-AUC:  {pr_auc:.6f}")

ROC-AUC: 0.999506
PR-AUC:  0.998063


In [27]:
# ============================================================
# 18. XGBOOST FEATURE IMPORTANCE
# ============================================================

# Get feature names after preprocessing
feature_names = preprocessor.get_feature_names_out()

# Get feature importance from the trained XGBoost model
importance_values = xgb_model.feature_importances_

# Create a DataFrame
feature_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": importance_values
})

# Sort from most important to least important
feature_importance = feature_importance.sort_values(
    by="importance",
    ascending=False
)

# Display top 20 features
print("Top 20 Most Important Features:")
display(feature_importance.head(20))

Top 20 Most Important Features:


,feature,importance
17,numerical__full_origin_balance_transfer,0.580593
22,numerical__origin_balance_change,0.213719
20,numerical__origin_balance_depleted,0.067020
3,categorical__type_PAYMENT,0.043340
14,numerical__origin_balance_utilization,0.022528
6,numerical__amount,0.012890
11,numerical__log_amount,0.012675
24,numerical__zero_amount_transaction,0.009090
8,numerical__newbalanceOrig,0.007435
7,numerical__oldbalanceOrg,0.005877


In [28]:
# ============================================================
# 19. FEATURE LEAKAGE / AVAILABILITY AUDIT
# ============================================================

audit_features = [
    "full_origin_balance_transfer",
    "origin_balance_change",
    "origin_balance_depleted",
    "origin_balance_utilization",
    "destination_balance_change",
    "amount_to_destination_balance",
    "zero_amount_transaction"
]

print("Feature Statistics:")
print("=" * 70)

for feature in audit_features:

    print(f"\n{feature}")
    print("-" * 70)

    print("Unique values:")
    print(df[feature].value_counts().head(10))

    print("\nFraud rate by feature:")
    print(
        df.groupby(feature)["isFraud"]
        .agg(["count", "mean"])
        .sort_values("mean", ascending=False)
        .head(10)
    )

Feature Statistics:

full_origin_balance_transfer
----------------------------------------------------------------------
Unique values:
full_origin_balance_transfer
0    6354602
1       8018
Name: count, dtype: int64

Fraud rate by feature:
                                count      mean
full_origin_balance_transfer                   
1                                8018  1.000000
0                             6354602  0.000031

origin_balance_change
----------------------------------------------------------------------
Unique values:
origin_balance_change
0.0      2089037
184.0        737
181.0        732
186.0        731
157.0        728
195.0        728
146.0        726
111.0        726
164.0        722
136.0        722
Name: count, dtype: int64

Fraud rate by feature:
                       count  mean
origin_balance_change             
210.92                     2   1.0
10000000.00                3   1.0
9593838.63                 2   1.0
403.56                     2   1.0
577770

In [29]:
# ============================================================
# 20. INSPECT FULL ORIGIN BALANCE TRANSFER
# ============================================================

full_transfer_df = df[ df["full_origin_balance_transfer"] == 1].copy()

print("Number of full-balance-transfer transactions:", len(full_transfer_df))

print("\nFraud count:")
print(full_transfer_df["isFraud"].value_counts())

print("\nFraud percentage:")
print(full_transfer_df["isFraud"].mean() * 100)

print("\nSample transactions:")
display(
    full_transfer_df[
        [
            "type",
            "amount",
            "oldbalanceOrg",
            "newbalanceOrig",
            "origin_balance_change",
            "origin_balance_depleted",
            "full_origin_balance_transfer",
            "isFraud"
        ]
    ].head(20)
)

Number of full-balance-transfer transactions: 8018

Fraud count:
isFraud
1    8018
Name: count, dtype: int64

Fraud percentage:
100.0

Sample transactions:


,type,amount,oldbalanceOrg,newbalanceOrig,origin_balance_change,origin_balance_depleted,full_origin_balance_transfer,isFraud
2,TRANSFER,181.00,181.00,0.0,181.00,1,1,1
3,CASH_OUT,181.00,181.00,0.0,181.00,1,1,1
251,TRANSFER,2806.00,2806.00,0.0,2806.00,1,1,1
252,CASH_OUT,2806.00,2806.00,0.0,2806.00,1,1,1
680,TRANSFER,20128.00,20128.00,0.0,20128.00,1,1,1
681,CASH_OUT,20128.00,20128.00,0.0,20128.00,1,1,1
969,TRANSFER,1277212.77,1277212.77,0.0,1277212.77,1,1,1
970,CASH_OUT,1277212.77,1277212.77,0.0,1277212.77,1,1,1
1115,TRANSFER,35063.63,35063.63,0.0,35063.63,1,1,1
1116,CASH_OUT,35063.63,35063.63,0.0,35063.63,1,1,1


In [30]:
# ============================================================
# 21. RULE-BASED FRAUD BASELINE
# ============================================================

# Simple rule:
# Flag a transaction as fraud when the entire origin balance
# is transferred and the origin balance becomes zero.

y_val_rule = ( X_val["full_origin_balance_transfer"] == 1).astype(int)

# Confusion matrix
rule_cm = confusion_matrix(y_val, y_val_rule)

print("Rule-Based Confusion Matrix:")
print(rule_cm)

# Extract confusion matrix values
rule_tn, rule_fp, rule_fn, rule_tp = rule_cm.ravel()

print("\nRule-Based Results:")
print(f"True Negatives  (TN): {rule_tn:,}")
print(f"False Positives (FP): {rule_fp:,}")
print(f"False Negatives (FN): {rule_fn:,}")
print(f"True Positives  (TP): {rule_tp:,}")

# Calculate precision and recall
rule_precision = rule_tp / (rule_tp + rule_fp)
rule_recall = rule_tp / (rule_tp + rule_fn)

print(f"\nPrecision: {rule_precision:.4f}")
print(f"Recall:    {rule_recall:.4f}")

Rule-Based Confusion Matrix:
[[953161      0]
 [    25   1207]]

Rule-Based Results:
True Negatives  (TN): 953,161
False Positives (FP): 0
False Negatives (FN): 25
True Positives  (TP): 1,207

Precision: 1.0000
Recall:    0.9797


In [31]:
# ============================================================
# 22. THRESHOLD ANALYSIS
# ============================================================

from sklearn.metrics import precision_score, recall_score, f1_score

In [33]:
thresholds = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]

threshold_results = []

for threshold in thresholds:

    y_threshold_pred = (y_val_proba >= threshold).astype(int)

    # Calculate metrices
    precision = precision_score(y_val, y_threshold_pred, zero_division=0)
    recall = recall_score(y_val, y_threshold_pred, zero_division=0)
    f1 = f1_score(y_val, y_threshold_pred, zero_division=0)

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    })

threshold_df = pd.DataFrame(threshold_results)

print("Threshold Analysis")

display(threshold_df)


Threshold Analysis


,Threshold,Precision,Recall,F1
0,0.1,0.992724,0.996753,0.994735
1,0.2,0.994332,0.996753,0.995541
2,0.3,0.995138,0.996753,0.995945
3,0.4,0.995138,0.996753,0.995945
4,0.5,0.996753,0.996753,0.996753
5,0.6,0.996753,0.996753,0.996753
6,0.7,0.997563,0.996753,0.997158
7,0.8,0.997563,0.996753,0.997158
8,0.9,0.999186,0.996753,0.997968


In [34]:
# ============================================================
# 23. FINAL TEST-SET EVALUATION
# ============================================================

# Generate fraud probabilities for the untouched test set
y_test_proba = xgb_model.predict_proba( X_test_processed)[:, 1]

# Use the selected validation threshold
final_threshold = 0.90

# Convert probabilities into fraud predictions
y_test_pred = ( y_test_proba >= final_threshold).astype(int)

# Confusion matrix
test_cm = confusion_matrix( y_test, y_test_pred)

print("Test Confusion Matrix:")
print(test_cm)

# Extract confusion matrix values
test_tn, test_fp, test_fn, test_tp = test_cm.ravel()

print("\nTest Results:")
print(f"True Negatives  (TN): {test_tn:,}")
print(f"False Positives (FP): {test_fp:,}")
print(f"False Negatives (FN): {test_fn:,}")
print(f"True Positives  (TP): {test_tp:,}")

# Classification metrics
test_precision = precision_score(y_test, y_test_pred, zero_division=0)

test_recall = recall_score(y_test, y_test_pred, zero_division=0)

test_f1= f1_score(y_test, y_test_pred, zero_division=0)

test_roc_auc = roc_auc_score(y_test, y_test_proba)

test_pr_auc = average_precision_score( y_test, y_test_proba)

print("\nTest Metrics:")
print(f"Precision: {test_precision:.6f}")
print(f"Recall:    {test_recall:.6f}")
print(f"F1 Score:  {test_f1:.6f}")
print(f"ROC-AUC:   {test_roc_auc:.6f}")
print(f"PR-AUC:    {test_pr_auc:.6f}")

Test Confusion Matrix:
[[953160      1]
 [     5   1227]]

Test Results:
True Negatives  (TN): 953,160
False Positives (FP): 1
False Negatives (FN): 5
True Positives  (TP): 1,227

Test Metrics:
Precision: 0.999186
Recall:    0.995942
F1 Score:  0.997561
ROC-AUC:   0.999341
PR-AUC:    0.998372


In [35]:
# ============================================================
# 24. TRAIN XGBOOST WITH EARLY STOPPING
# ============================================================

xgb_early = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=30
)

xgb_early.fit(
    X_train_processed,
    y_train,
    eval_set=[
        (X_val_processed, y_val)
    ],
    verbose=50
)

print("\nBest iteration:", xgb_early.best_iteration)
print("Best validation PR-AUC:", xgb_early.best_score)

[0]	validation_0-aucpr:0.39544
[50]	validation_0-aucpr:0.99862
[70]	validation_0-aucpr:0.99877

Best iteration: 40
Best validation PR-AUC: 0.9988539565005358


In [36]:
# ============================================================
# 25. COMPARE ORIGINAL VS EARLY-STOPPED XGBOOST
# ============================================================

# Generate validation probabilities
y_val_proba_original = xgb_model.predict_proba(
    X_val_processed
)[:, 1]

y_val_proba_early = xgb_early.predict_proba(
    X_val_processed
)[:, 1]


# ------------------------------------------------------------
# ORIGINAL MODEL
# ------------------------------------------------------------

original_pr_auc = average_precision_score(
    y_val,
    y_val_proba_original
)

original_roc_auc = roc_auc_score(
    y_val,
    y_val_proba_original
)


# ------------------------------------------------------------
# EARLY-STOPPED MODEL
# ------------------------------------------------------------

early_pr_auc = average_precision_score(
    y_val,
    y_val_proba_early
)

early_roc_auc = roc_auc_score(
    y_val,
    y_val_proba_early
)


# ------------------------------------------------------------
# THRESHOLD = 0.90
# ------------------------------------------------------------

threshold = 0.90

y_val_pred_original = (
    y_val_proba_original >= threshold
).astype(int)

y_val_pred_early = (
    y_val_proba_early >= threshold
).astype(int)


original_precision = precision_score(
    y_val,
    y_val_pred_original,
    zero_division=0
)

original_recall = recall_score(
    y_val,
    y_val_pred_original,
    zero_division=0
)

original_f1 = f1_score(
    y_val,
    y_val_pred_original,
    zero_division=0
)


early_precision = precision_score(
    y_val,
    y_val_pred_early,
    zero_division=0
)

early_recall = recall_score(
    y_val,
    y_val_pred_early,
    zero_division=0
)

early_f1 = f1_score(
    y_val,
    y_val_pred_early,
    zero_division=0
)


# ------------------------------------------------------------
# COMPARISON TABLE
# ------------------------------------------------------------

comparison = pd.DataFrame({
    "Metric": [
        "PR-AUC",
        "ROC-AUC",
        "Precision @ 0.90",
        "Recall @ 0.90",
        "F1 @ 0.90"
    ],
    "Original XGBoost": [
        original_pr_auc,
        original_roc_auc,
        original_precision,
        original_recall,
        original_f1
    ],
    "Early-Stopped XGBoost": [
        early_pr_auc,
        early_roc_auc,
        early_precision,
        early_recall,
        early_f1
    ]
})

print("MODEL COMPARISON")
print("=" * 70)

display(comparison)

MODEL COMPARISON


,Metric,Original XGBoost,Early-Stopped XGBoost
0,PR-AUC,0.998063,0.998851
1,ROC-AUC,0.999506,0.999518
2,Precision @ 0.90,0.999186,1.000000
3,Recall @ 0.90,0.996753,0.996753
4,F1 @ 0.90,0.997968,0.998374


In [37]:
# ============================================================
# 26. FINAL MODEL CONFIGURATION
# ============================================================

final_model = xgb_early

final_threshold = 0.90

print("FINAL MODEL")
print("=" * 60)

print("Model: XGBoost Classifier")
print("Early Stopping: Enabled")
print("Best Iteration:", final_model.best_iteration)
print("Best Validation PR-AUC:", final_model.best_score)

print("\nDecision Threshold:", final_threshold)

print("\nModel Parameters:")
print("max_depth:", final_model.max_depth)
print("learning_rate:", final_model.learning_rate)
print("subsample:", final_model.subsample)
print("colsample_bytree:", final_model.colsample_bytree)
print("scale_pos_weight:", final_model.scale_pos_weight)

FINAL MODEL
Model: XGBoost Classifier
Early Stopping: Enabled
Best Iteration: 40
Best Validation PR-AUC: 0.9988539565005358

Decision Threshold: 0.9

Model Parameters:
max_depth: 6
learning_rate: 0.1
subsample: 0.8
colsample_bytree: 0.8
scale_pos_weight: 773.7145590537485


In [38]:
# ============================================================
# 27. FINAL MODEL TEST-SET EVALUATION
# ============================================================

# Generate fraud probabilities using the FINAL model
y_test_proba_final = final_model.predict_proba( X_test_processed)[:, 1]


# Apply final decision threshold
y_test_pred_final = ( y_test_proba_final >= final_threshold).astype(int)

# ------------------------------------------------------------
# CONFUSION MATRIX
# ------------------------------------------------------------

final_cm = confusion_matrix(y_test, y_test_pred_final)

print("Final Test Confusion Matrix:")
print(final_cm)

# Extract confusion matrix values
final_tn, final_fp, final_fn, final_tp = final_cm.ravel()

print("\nFinal Test Results:")
print(f"True Negatives  (TN): {final_tn:,}")
print(f"False Positives (FP): {final_fp:,}")
print(f"False Negatives (FN): {final_fn:,}")
print(f"True Positives  (TP): {final_tp:,}")

# ------------------------------------------------------------
# FINAL METRICS
# ------------------------------------------------------------

final_precision = precision_score(y_test, y_test_pred_final, zero_division=0)

final_recall = recall_score(y_test, y_test_pred_final, zero_division=0)

final_f1 = f1_score(y_test, y_test_pred_final, zero_division=0)

final_roc_auc = roc_auc_score( y_test, y_test_proba_final )

final_pr_auc = average_precision_score(y_test, y_test_proba_final )

# ------------------------------------------------------------
# DISPLAY FINAL METRICS
# ------------------------------------------------------------

print("\nFINAL TEST METRICS")
print("=" * 60)

print(f"Precision: {final_precision:.6f}")
print(f"Recall:    {final_recall:.6f}")
print(f"F1 Score:  {final_f1:.6f}")
print(f"ROC-AUC:   {final_roc_auc:.6f}")
print(f"PR-AUC:    {final_pr_auc:.6f}")

Final Test Confusion Matrix:
[[953161      0]
 [     5   1227]]

Final Test Results:
True Negatives  (TN): 953,161
False Positives (FP): 0
False Negatives (FN): 5
True Positives  (TP): 1,227

FINAL TEST METRICS
Precision: 1.000000
Recall:    0.995942
F1 Score:  0.997967
ROC-AUC:   0.999841
PR-AUC:    0.998174


In [39]:
# ============================================================
# 28. FINAL MODEL & PREPROCESSOR AUDIT
# ============================================================

print("FINAL MODEL AUDIT")
print("=" * 60)


# ------------------------------------------------------------
# 1. FINAL MODEL
# ------------------------------------------------------------

print("\n[1] Final Model")
print("Model type:", type(final_model).__name__)
print("Best iteration:", final_model.best_iteration)
print("Best validation PR-AUC:", final_model.best_score)


# ------------------------------------------------------------
# 2. DECISION THRESHOLD
# ------------------------------------------------------------

print("\n[2] Decision Threshold")
print("Final threshold:", final_threshold)


# ------------------------------------------------------------
# 3. PREPROCESSOR
# ------------------------------------------------------------

print("\n[3] Preprocessor")
print("Categorical features:", categorical_features)
print("Number of numerical features:", len(numerical_features))


# ------------------------------------------------------------
# 4. FEATURE DIMENSIONS
# ------------------------------------------------------------

print("\n[4] Feature Dimensions")
print("Raw feature count:", X_train.shape[1])
print("Processed feature count:", X_train_processed.shape[1])


# ------------------------------------------------------------
# 5. PROCESSED FEATURE NAMES
# ------------------------------------------------------------

feature_names = preprocessor.get_feature_names_out()

print("\n[5] Processed Features")
print("Number of feature names:", len(feature_names))

print("\nFeature names:")
for i, feature in enumerate(feature_names, start=1):
    print(f"{i:02d}. {feature}")


# ------------------------------------------------------------
# 6. IDENTIFIER CHECK
# ------------------------------------------------------------

print("\n[6] Identifier Check")

identifier_columns = ["nameOrig", "nameDest"]

remaining_identifiers = [
    col for col in identifier_columns
    if col in X_train.columns
]

if len(remaining_identifiers) == 0:
    print("PASS — Identifier columns are excluded.")
else:
    print("WARNING — Identifier columns found:", remaining_identifiers)


# ------------------------------------------------------------
# 7. FINAL TEST METRICS
# ------------------------------------------------------------

print("\n[7] Final Test Metrics")

print(f"Precision: {final_precision:.6f}")
print(f"Recall:    {final_recall:.6f}")
print(f"F1 Score:  {final_f1:.6f}")
print(f"ROC-AUC:   {final_roc_auc:.6f}")
print(f"PR-AUC:    {final_pr_auc:.6f}")


print("\n" + "=" * 60)
print("FINAL AUDIT COMPLETE")

FINAL MODEL AUDIT

[1] Final Model
Model type: XGBClassifier
Best iteration: 40
Best validation PR-AUC: 0.9988539565005358

[2] Decision Threshold
Final threshold: 0.9

[3] Preprocessor
Categorical features: ['type']
Number of numerical features: 20

[4] Feature Dimensions
Raw feature count: 21
Processed feature count: 25

[5] Processed Features
Number of feature names: 25

Feature names:
01. categorical__type_CASH_IN
02. categorical__type_CASH_OUT
03. categorical__type_DEBIT
04. categorical__type_PAYMENT
05. categorical__type_TRANSFER
06. numerical__step
07. numerical__amount
08. numerical__oldbalanceOrg
09. numerical__newbalanceOrig
10. numerical__oldbalanceDest
11. numerical__newbalanceDest
12. numerical__log_amount
13. numerical__hour
14. numerical__day
15. numerical__origin_balance_utilization
16. numerical__origin_balance_zero
17. numerical__destination_balance_zero
18. numerical__full_origin_balance_transfer
19. numerical__destination_previous_transactions
20. numerical__origin_

In [40]:
# ============================================================
# 29. SAVE FINAL MODEL AND PREPROCESSOR
# ============================================================

import os
import joblib

# ------------------------------------------------------------
# 1. DEFINE MODEL DIRECTORY
# ------------------------------------------------------------

MODEL_DIR = "../models"

os.makedirs(MODEL_DIR, exist_ok=True)

# ------------------------------------------------------------
# 2. SAVE FINAL XGBOOST MODEL
# ------------------------------------------------------------

MODEL_PATH = os.path.join(
    MODEL_DIR,
    "xgboost_fraud_model.pkl"
)

joblib.dump(
    final_model,
    MODEL_PATH
)

# ------------------------------------------------------------
# 3. SAVE PREPROCESSOR
# ------------------------------------------------------------

PREPROCESSOR_PATH = os.path.join(
    MODEL_DIR,
    "preprocessor.pkl"
)

joblib.dump(
    preprocessor,
    PREPROCESSOR_PATH
)

# ------------------------------------------------------------
# 4. CONFIRM FILES
# ------------------------------------------------------------

print("MODEL ARTIFACTS SAVED")
print("=" * 60)

print("Model:")
print(MODEL_PATH)

print("\nPreprocessor:")
print(PREPROCESSOR_PATH)

print("\nFiles exist:")
print("Model:", os.path.exists(MODEL_PATH))
print("Preprocessor:", os.path.exists(PREPROCESSOR_PATH))

MODEL ARTIFACTS SAVED
Model:
../models\xgboost_fraud_model.pkl

Preprocessor:
../models\preprocessor.pkl

Files exist:
Model: True
Preprocessor: True


In [41]:
# ============================================================
# 30. VERIFY SAVED MODEL ARTIFACTS
# ============================================================

import joblib
import numpy as np

# -----------------------------------------------------------
# 1. LOAD SAVED ARTIFACTS
# ------------------------------------------------------------

loaded_model = joblib.load(MODEL_PATH)

loaded_preprocessor = joblib.load(PREPROCESSOR_PATH)

# ------------------------------------------------------------
# 2. VERIFY MODEL TYPE
# ------------------------------------------------------------

print("LOADED ARTIFACT VERIFICATION")
print("=" * 60)

print("\nLoaded model type:")
print(type(loaded_model).__name__)

print("\nLoaded preprocessor type:")
print(type(loaded_preprocessor).__name__)

# ------------------------------------------------------------
# 3. VERIFY PREPROCESSING
# ------------------------------------------------------------

X_test_loaded = loaded_preprocessor.transform(X_test)

print("\nProcessed test shape:")
print(X_test_loaded.shape)

# ------------------------------------------------------------
# 4. GENERATE PREDICTIONS
# ------------------------------------------------------------

loaded_proba = loaded_model.predict_proba(
    X_test_loaded
)[:, 1]

# ------------------------------------------------------------
# 5. COMPARE WITH ORIGINAL PREDICTIONS
# ------------------------------------------------------------

max_difference = np.max(
    np.abs(
        loaded_proba - y_test_proba_final
    )
)

print("\nMaximum probability difference:")
print(max_difference)


# ------------------------------------------------------------
# 6. VERIFY PREDICTION CONSISTENCY
# ------------------------------------------------------------

loaded_predictions = (
    loaded_proba >= final_threshold
).astype(int)

prediction_match = np.array_equal(
    loaded_predictions,
    y_test_pred_final
)

print("\nPredictions identical:", prediction_match)


# ------------------------------------------------------------
# 7. FINAL STATUS
# ------------------------------------------------------------

print("\n" + "=" * 60)

if prediction_match:
    print("PASS — Saved artifacts reproduce the original predictions.")
else:
    print("WARNING — Prediction mismatch detected.")

LOADED ARTIFACT VERIFICATION

Loaded model type:
XGBClassifier

Loaded preprocessor type:
ColumnTransformer

Processed test shape:
(954393, 25)

Maximum probability difference:
0.0

Predictions identical: True

PASS — Saved artifacts reproduce the original predictions.


In [42]:
# ============================================================
# 31. FINAL MODEL PERFORMANCE SUMMARY
# ============================================================

final_summary = pd.DataFrame({
    "Metric": [
        "Decision Threshold",
        "Best Iteration",
        "Precision",
        "Recall",
        "F1 Score",
        "ROC-AUC",
        "PR-AUC",
        "True Negatives",
        "False Positives",
        "False Negatives",
        "True Positives"
    ],
    "Value": [
        final_threshold,
        final_model.best_iteration,
        final_precision,
        final_recall,
        final_f1,
        final_roc_auc,
        final_pr_auc,
        final_tn,
        final_fp,
        final_fn,
        final_tp
    ]
})


print("FINAL MODEL PERFORMANCE SUMMARY")
print("=" * 60)

display(final_summary)

FINAL MODEL PERFORMANCE SUMMARY


,Metric,Value
0,Decision Threshold,0.900000
1,Best Iteration,40.000000
2,Precision,1.000000
3,Recall,0.995942
4,F1 Score,0.997967
5,ROC-AUC,0.999841
6,PR-AUC,0.998174
7,True Negatives,953161.000000
8,False Positives,0.000000
9,False Negatives,5.000000


In [43]:
# ============================================================
# 32. FINAL MODEL FEATURE IMPORTANCE
# ============================================================

# Get processed feature names
final_feature_names = preprocessor.get_feature_names_out()

# Get feature importance from final model
final_importance_values = final_model.feature_importances_


# ------------------------------------------------------------
# CREATE FEATURE IMPORTANCE DATAFRAME
# ------------------------------------------------------------

final_feature_importance = pd.DataFrame({
    "feature": final_feature_names,
    "importance": final_importance_values
})


# Sort from most to least important
final_feature_importance = final_feature_importance.sort_values(
    by="importance",
    ascending=False
).reset_index(drop=True)


# ------------------------------------------------------------
# DISPLAY TOP FEATURES
# ------------------------------------------------------------

print("FINAL MODEL — FEATURE IMPORTANCE")
print("=" * 60)

display(
    final_feature_importance.head(20)
)


# ------------------------------------------------------------
# TOP 10 FEATURES
# ------------------------------------------------------------

print("\nTop 10 Features:")

for i, row in final_feature_importance.head(10).iterrows():
    print(
        f"{i + 1:02d}. "
        f"{row['feature']} → "
        f"{row['importance']:.6f}"
    )

FINAL MODEL — FEATURE IMPORTANCE


,feature,importance
0,numerical__full_origin_balance_transfer,0.662086
1,numerical__origin_balance_change,0.177740
2,numerical__origin_balance_depleted,0.071614
3,categorical__type_PAYMENT,0.023986
4,numerical__origin_balance_utilization,0.019503
5,numerical__log_amount,0.010206
6,numerical__amount,0.010036
7,numerical__zero_amount_transaction,0.003166
8,numerical__destination_balance_zero,0.002823
9,numerical__oldbalanceOrg,0.002670



Top 10 Features:
01. numerical__full_origin_balance_transfer → 0.662086
02. numerical__origin_balance_change → 0.177740
03. numerical__origin_balance_depleted → 0.071614
04. categorical__type_PAYMENT → 0.023986
05. numerical__origin_balance_utilization → 0.019503
06. numerical__log_amount → 0.010206
07. numerical__amount → 0.010036
08. numerical__zero_amount_transaction → 0.003166
09. numerical__destination_balance_zero → 0.002823
10. numerical__oldbalanceOrg → 0.002670


In [44]:
# ============================================================
# FINAL MODEL DOCUMENTATION
# ============================================================

print("=" * 70)
print("FINANCIAL TRANSACTION RISK INTELLIGENCE")
print("FINAL SUPERVISED MODEL SUMMARY")
print("=" * 70)


# ------------------------------------------------------------
# 1. FINAL MODEL
# ------------------------------------------------------------

print("\n1. FINAL MODEL")
print("-" * 70)

print("Algorithm: XGBoost Classifier")
print("Early stopping: Enabled")
print("Best iteration:", final_model.best_iteration)
print("Decision threshold:", final_threshold)


# ------------------------------------------------------------
# 2. FINAL TEST PERFORMANCE
# ------------------------------------------------------------

print("\n2. FINAL TEST PERFORMANCE")
print("-" * 70)

print(f"Precision : {final_precision:.4f}")
print(f"Recall    : {final_recall:.4f}")
print(f"F1 Score  : {final_f1:.4f}")
print(f"ROC-AUC   : {final_roc_auc:.4f}")
print(f"PR-AUC    : {final_pr_auc:.4f}")


# ------------------------------------------------------------
# 3. CONFUSION MATRIX RESULTS
# ------------------------------------------------------------

print("\n3. CONFUSION MATRIX")
print("-" * 70)

print(f"True Negatives  : {final_tn:,}")
print(f"False Positives : {final_fp:,}")
print(f"False Negatives : {final_fn:,}")
print(f"True Positives  : {final_tp:,}")

# ------------------------------------------------------------
# 4. FEATURE IMPORTANCE
# ------------------------------------------------------------

print("\n4. TOP 5 FEATURES")
print("-" * 70)

for i, row in final_feature_importance.head(5).iterrows():
    print(
        f"{i + 1}. {row['feature']} "
        f"({row['importance'] * 100:.2f}%)"
    )


# ------------------------------------------------------------
# 5. KEY BUSINESS INTERPRETATION
# ------------------------------------------------------------

print("\n5. BUSINESS INTERPRETATION")
print("-" * 70)

print(
    "The model is highly effective at distinguishing fraudulent "
    "transactions from legitimate transactions."
)

print(
    "At a 0.90 decision threshold, the final model produced "
    "zero false positives and detected 1,227 fraudulent "
    "transactions in the held-out test set."
)

print(
    "The model relies heavily on transaction-level origin-account "
    "balance behavior, particularly full-balance transfers, "
    "balance changes, and balance depletion."
)


# ------------------------------------------------------------
# 6. IMPORTANT LIMITATION
# ------------------------------------------------------------

print("\n6. MODEL LIMITATION")
print("-" * 70)

print(
    "The dominant full_origin_balance_transfer feature captures "
    "a very strong structural pattern in the PaySim dataset."
)

print(
    "Because some engineered features depend on post-transaction "
    "balance information, their availability should be evaluated "
    "before deploying the model in a real-time authorization system."
)

print(
    "Therefore, the reported performance should be interpreted "
    "as performance on the PaySim dataset rather than guaranteed "
    "real-world fraud detection performance."
)


# ------------------------------------------------------------
# 7. FINAL STATUS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SUPERVISED MODELING PHASE COMPLETE")
print("=" * 70)

FINANCIAL TRANSACTION RISK INTELLIGENCE
FINAL SUPERVISED MODEL SUMMARY

1. FINAL MODEL
----------------------------------------------------------------------
Algorithm: XGBoost Classifier
Early stopping: Enabled
Best iteration: 40
Decision threshold: 0.9

2. FINAL TEST PERFORMANCE
----------------------------------------------------------------------
Precision : 1.0000
Recall    : 0.9959
F1 Score  : 0.9980
ROC-AUC   : 0.9998
PR-AUC    : 0.9982

3. CONFUSION MATRIX
----------------------------------------------------------------------
True Negatives  : 953,161
False Positives : 0
False Negatives : 5
True Positives  : 1,227

4. TOP 5 FEATURES
----------------------------------------------------------------------
1. numerical__full_origin_balance_transfer (66.21%)
2. numerical__origin_balance_change (17.77%)
3. numerical__origin_balance_depleted (7.16%)
4. categorical__type_PAYMENT (2.40%)
5. numerical__origin_balance_utilization (1.95%)

5. BUSINESS INTERPRETATION
-----------------------